# Codegen
CodeGen is a family of autoregressive language models for program synthesis from the paper: A Conversational Paradigm for Program Synthesis by Erik Nijkamp, Bo Pang, Hiroaki Hayashi, Lifu Tu, Huan Wang, Yingbo Zhou, Silvio Savarese, Caiming Xiong. 

In this Notebook we are going to load the codegen-2B-mono model where:
* **2B** stands for the amount of parameters of the model
* **mono** stands on what is trained on, in this case mono means that the model was finetuned for python code generation only.

In [ ]:
# PIP INSTALLS
# These will eventually be already installed in the container, so there should be no pip install inside the notebook
!pip install accelerate
#Transformers needs to be updated
!pip install  transformers
#Install our qaic apis
!python3 -m pip install /opt/qti-aic/dev/lib/x86_64/qaic-0.0.1-py3-none-any.whl
!pip install torch
!pip install numpy
!pip install onnx

In [ ]:
!pip install tensorflow

## Imports

In [10]:
import qaic
import numpy as np
import tensorflow as tf
import os
import shutil
from pathlib import Path
import torch
from torch.onnx import export
import onnx
from packaging import version

## Transformers from Huggingface

We are going to use the Transformer library from huggingface.

Transformers offers the Causal language modeling for text generation where the model predicts the next token given the current sequence of tokens.

In [1]:
from transformers import CodeGenForCausalLM, AutoTokenizer

/local/mnt/workspace/fbuoncom/venv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2023-06-16 13:02:46.783159: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-06-16 13:02:48.418351: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2023-06-16 13:02:48.423886: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-06-16 13:03:04.974444: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


## Download the model

In [ ]:
# # Load the model and tokenizer
checkpoint = "Salesforce/codegen-2B-mono"
model = CodeGenForCausalLM.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

model.save_pretrained("./cd/model",max_shard_size='20GB') # So that the model is not split in 2
tokenizer.save_pretrained("./cd/tokenizer")
del model
del tokenizer

## Load the saved model
We are going to load the tokenizer only, the actual model will be loaded during ONNX conversion

In [3]:
tokenizer = AutoTokenizer.from_pretrained('./cd/tokenizer')

## First generation of the ONNX
Here we create the ONNX.
From the forward pass of the model CodeGenForCausalLM, we can understand what is the **input_ids** required by the model.


        input_ids (`torch.LongTensor` of shape `(batch_size, sequence_length)`):
            Indices of input sequence tokens in the vocabular
            
            
These are the result of the tokenizer component.
-ids)


In [4]:
@torch.no_grad()
def convert_models(model_path: str, output_path: str, opset: int):
    print("Loading the model")
    model = CodeGenForCausalLM.from_pretrained(model_path)
    output_path = Path(output_path)
    output_model_path = output_path / "model.onnx"
    print("Loaded the model")
    
    # Dicts
    input_names = ["input_ids"]
    output_names = ["output"]
    dynamic_axes = {
        "input_ids": {0: "batch_size", 1: "sequence_length"},
        "output": {0: "batch_size", 1: "sequence_length"},
    }
    
    print("Creating the dummy input")
    text = "def dummy_input:"
    # Make the pad token equal to the eos token
    tokenizer.pad_token = tokenizer.eos_token
    tokens = tokenizer(text,
                       padding='max_length',
                       max_length=128,
                       return_tensors="pt")
    
    print("Created the dummy input")
    print("Starting the export")
    torch.onnx.export(
        model,
        tokens['input_ids'],
        f=output_model_path.as_posix(),
        input_names=input_names,
        output_names=output_names,
        dynamic_axes=dynamic_axes,
        do_constant_folding=True,
        opset_version=opset
    )
    print("Finished the export")
    
    output_model_path = str(output_model_path.absolute().as_posix())
    output_model_dir = os.path.dirname(output_model_path)
    model = onnx.load(output_model_path)
    # clean up existing tensor files
    shutil.rmtree(output_model_dir)
    os.mkdir(output_model_dir)
    
    print("Started the squashing of the model")
    onnx.save_model(
        model,
        output_model_path,
        save_as_external_data=True,
        all_tensors_to_one_file=True,
        location="weights.pb",
        convert_attribute=False,
    )
    print("Squashed model")
    del model

In [5]:
# Convert the UNet with ONNX opset 14
convert_models('./cd/model','./cd/model_onnx',14)
print("Finished")

Loading the model
Loaded the model
Creating the dummy input
Created the dummy input
Starting the export
> /local/mnt/workspace/fbuoncom/venv/lib/python3.8/site-packages/transformers/models/codegen/modeling_codegen.py(459)forward()
    457         return_dict = return_dict if return_dict is not None else self.config.use_return_dict
    458         import pdb; pdb.set_trace()
--> 459         if input_ids is not None and inputs_embeds is not None:
    460             raise ValueError("You cannot specify both input_ids and inputs_embeds at the same time")
    461         elif input_ids is not None:



ipdb>  c


/local/mnt/workspace/fbuoncom/venv/lib/python3.8/site-packages/transformers/models/codegen/modeling_codegen.py:147: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  mask_value = torch.tensor(mask_value, dtype=attn_weights.dtype).to(attn_weights.device)


======== Diagnostic Run torch.onnx.export version 2.1.0.dev20230612+cpu ========
verbose: False, log level: 40
======================= 0 NONE 0 NOTE 0 WARNING 0 ERROR ========================

Finished the export
Started the squashing of the model
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
Squashed model
Finished


## Use of the HL Python API

We will use the qaic.Session to create our qpc.

Let's analyze the various parameters passed to the qaic.Session() constructor.

| Parameter | Explanation |
| --- | --- |
| **dev_id** | The device id we are going to use for our calculation|
| **aic_num_cores** | 	Number of aic cores to be used for inference |
| **convert_to_fp16** | Run all floating-point in fp16 |

In [4]:
onnx_model_path="./cd/model_onnx/model.onnx"
qpc_model_path="./cd/model_qpc/programqpc.bin"

In [5]:
qaic_sess = qaic.Session(onnx_model_path,
                         dev_id=0,
                         aic_num_cores=14,
                         convert_to_fp16=True,
                         onnx_define_symbol={"batch_size":1,
                                             "sequence_length":128},
                         output_dir="./cd/model_qpc/"
                        )

Options yaml file available at:  ./cd/model_qpc//options.yaml


In [6]:
# Free memory, this is after the first compilation of the qpc
del qaic_sess

## Prepare our input text

Now we have our Session ready we can run inference, let's create an example prompt.

In [4]:
text = "# Class definition of a Dog"
tokens_no_pad = tokenizer(text,
                   return_tensors="pt")['input_ids']
initial_token_size=int(tokens_no_pad.shape[1])

In [5]:
tokens_no_pad

tensor([[   2, 5016, 6770,  286,  257, 8532]])

In [6]:
tokenizer.pad_token = tokenizer.eos_token
tokens = tokenizer(text,
                   padding='max_length',
                   max_length=128,
                   return_tensors="pt")

In [7]:
tokens['input_ids']=tokens['input_ids'].cpu().numpy()
starting_inputs = tokens['input_ids']

In [8]:
starting_inputs

array([[    2,  5016,  6770,   286,   257,  8532, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
        50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256, 50256,
      

## Load the QPC on the card

In [11]:
qpc_model_path = "./cd/model_qpc/programqpc.bin"
yaml_options_path = "./cd/model_qpc/options.yaml"
qaic_sess = qaic.Session(qpc_model_path, options_path=yaml_options_path)

## Inference
This is a small example of a possible loop that takes the tokens and select each next token based on the most probable one.

This loop is deterministic, it will produce the same result given the same prompt, it is called **greedy decoding**.

This beahviour can be modified with the idea of sampling from the logits. 
Currently the most used sample teqnciques are Temperature sampling or Top-K sampling.


In [12]:
def generate_qaic_output(qaic_sess, original_input_ids, initial_token_size, max_new_tokens):
    cur_len = initial_token_size
    input_ids = original_input_ids

    while cur_len < max_new_tokens:
        logits = qaic_sess.run({"input_ids":input_ids})['output']
        logits = logits.reshape(1,128,51200)
        
        # Get the logits of the last token
        next_token_logits = logits[:, cur_len-1, :]

        # Choose the token with the highest probability
        next_token = np.argmax(next_token_logits, axis=1)[:, np.newaxis]

        # Append next_token at the end of input_ids
        input_ids[0][cur_len]=next_token
        cur_len += 1
            
    return input_ids

In [20]:
from transformers import top_k_top_p_filtering
from torch.nn import functional as F
def generate_qaic_output_top_k(qaic_sess, original_input_ids, initial_token_size, max_new_tokens):
    cur_len = initial_token_size
    input_ids = original_input_ids

    while cur_len < max_new_tokens:
        logits = qaic_sess.run({"input_ids":input_ids})['output']
        logits = logits.reshape(1,128,51200)
        
        # Get the logits of the last token
        next_token_logits = logits[:, cur_len-1, :]

        # Keep top 30 logits at max; stop if cumulative probability >= 1.0.
        top_logits = top_k_top_p_filtering(torch.Tensor(next_token_logits), top_k=100, top_p=1.0)
        
        # Softmax the logits into probabilities
        probabilities = F.softmax(top_logits, dim=-1)
        
        # Generate next token
        next_token = torch.multinomial(probabilities, num_samples=1)
        
        # Append next_token at the end of input_ids
        input_ids[0][cur_len]=next_token
        cur_len += 1
            
    return input_ids

### Greedy Decoding Output

In [14]:
output_tokens = generate_qaic_output(qaic_sess,starting_inputs,initial_token_size,128)

qaic: WARNING: Acitvating network, this will add up to time of first inference


We now have the output in token form, we are going to use the decode function of the tokenizer to get actual text

In [15]:
decoded_string = tokenizer.decode(output_tokens[0])

In [16]:
print(decoded_string)

# Class definition of a Dog
#
# Author: David Fisher
# Course: CS151
# Date Created: 2020-02-20
# Date Revised: 2020-02-20

class Dog:
    """
    Class Dog that represents a dog.
    """

    def __init__(self, name, age, breed, spots):
        """
        Constructor for Dog class.
        :param name: (str) name of dog
        :param age: (int) age of dog
        :param breed: (str) breed of dog
        :param


### Top-k Decoding Output

In [ ]:
for i in range(4):
    output_tokens = generate_qaic_output_top_k(qaic_sess,starting_inputs,initial_token_size,128)
    decoded_string = tokenizer.decode(output_tokens[0])
    print(decoded_string)

# Class definition of a Dog object
#

class Dog:
    def __init__(self, name, legs, coatColor, isMuddy):
        self.__name = name
        self.__legs = legs
        self.__coatColor = coatColor
        self.__isMuddy = isMuddy

    def getName(self):
        """Return the dog's name."""
        return self.__name

    def getLegs(self):
        """Return
